# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [98]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [99]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2026, 1, 1)
END_DATE   = datetime.date(2026, 1, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-5% wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 1, 3)

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [100]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
# [:1] keeps execution fast; increase the slice to load more markets.
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 1 market file(s)
Unique markets (all):  89,622
Unique markets (filtered 2026-01-01 → 2026-01-05): 12,582


,end_date_iso,condition_id,market_json
0,2026-01-01T00:00:00Z,0x006ce0742c3891c396aead079e563c5d58c4eae2f9b05d910ed45110980290ad,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-10-20T21:13:17Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x006ce0742c3891c396aead079e563c5d58c4eae2f9b05d910ed45110980290ad"",""question_id"":""0x7cf043f4f8d4d781bdcedc859c3c8360936e8d2846b6c136bf722e192da06825"",""question"":""Will Rasmr win the Synthetix trading competition?"",""description"":""This market will resolve to the listed participant with the highest PnL at the end of the Synthetix trading competition starting on October 20, 2025.\n\nThe resolution source will be the Synthetix Leaderboard on https://www.synthetix.io/.\n\nIf two or more participants are tied for the highest PnL at the time of resolution, the market will resolve to the participant whose display name appears first in alphabetical order on the official Synthetix leaderboard.\n\nIf the competition is not completed by December 31, 2025, 11:59 PM ET, or if the official leaderboard or results are unavailable, the market will remain open until reliable data is published by Synthetix. If no such data is available by December 31, 2025, the market will resolve to “Other.”\n\nNote: The official competition includes 100 participants, but this market will resolve solely based on a selected group of KOL participants. Traders who are not eligible KOLs will not be considered. E.g. an eligible KOL will be considered to have won if they are the top trader out of eligible KOLs regardless of if one of the non KOL participants finishes ahead of them. \n\nAdditional KOLs may be added as options later to reflect the full set of eligible KOLs."",""market_slug"":""will-rasmr-win-the-synthetix-trading-competition-933"",""end_date_iso"":""2026-01-01T00:00:00Z"",""game_start_time"":null,""seconds_delay"":0,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":true,""neg_risk"":true,""neg_risk_market_id"":""0x7cf043f4f8d4d781bdcedc859c3c8360936e8d2846b6c136bf722e192da06800"",""neg_risk_request_id"":""0xda26edfbf8e42092e0693c8f4cedd5319c371ff9bd8650f2d7d7b00b19ce6fa2"",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/who-will-win-the-synthetix-trading-competition-TuVGkGEsLmbc.png"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/who-will-win-the-synthetix-trading-competition-TuVGkGEsLmbc.png"",""rewards"":{""rates"":null,""min_size"":50,""max_spread"":3.5},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""67487832607016995738972990163663499867372390570812445168484453192396152205498"",""outcome"":""Yes"",""price"":0,""winner"":false},{""token_id"":""99393257971182825969115164966386446587421508634202783327226489236020763806664"",""outcome"":""No"",""price"":1,""winner"":true}],""tags"":[""Crypto""]}"
1,2026-01-01T00:00:00Z,0x008add40355561724afbce56f68f1aa4ca4d83d8bdba3ed397f03a2743b47dd9,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-12-31T14:01:53Z"",""minimum_order_size"":5,""minimum_tick_size"":0.01,""condition_id"":""0x008add40355561724afbce56f68f1aa4ca4d83d8bdba3ed397f03a2743b47dd9"",""question_id"":""0x9c4aa152472d5b3332f1760add02bb13dc6804e1fe4021e35d8a3ecd8d1a4e29"",""question"":""Bitcoin Up or Down - January 1, 9:00AM-9:15AM ET"",""description"":""This market will resolve to \""Up\"" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to \""Down\"".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that this market is about the price according to Chainlink data stream BTC/

In [101]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Token lookup entries: 25,139
Market meta rows:     12,582


## 2 · Process partitioned per-side trades and filter to in-range markets

Trades are already stored as wallet-perspective rows (`wallet`, `side`) and
`TRADES_DIR` is partitioned, so we process each parquet shard independently to
avoid loading all fills into memory at once. For each shard we:
1. Inner-join with token resolution (`token_winner`, `final_price`)
2. Build `final_value_usdc = quantity × final_price`
3. Aggregate to wallet × market × timestamp and accumulate global results


In [102]:
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade parquet file(s)")

sample_raw = pd.read_parquet(trade_files)
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Sample shard date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)


Found 16 trade parquet file(s)
Sample shard rows (0.parquet): 93,357,019
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Sample shard date range: 2025-01-01 → 2026-03-11


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,wallet,side,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,78753205165658130534456351077321496563862891920229742523513427553266682271361,No,0x126f15c7bd21bf5bf136f3629e10990dc0677137,BUY,...,1,10.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
1,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,BUY,...,1,10.0,True,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",SELL,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
2,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38fce30001acc0181bf112,1531,66159089,1735689997,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x260d1ae03c985f7c78ad286b5da14940b4739679,BUY,...,1,30.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,Yes,58665055277986534416719939405914621458010137331379938342097742389618466217100,0.37,11.1


In [103]:
# ── build token resolution DataFrame from lookup ─────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

def enrich_trade_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Attach settlement columns and computed datetime/value to one shard."""
    if chunk.empty:
        return chunk

    c = chunk.copy()
    c["token_id"] = c["token_id"].astype(str)
    c = c.merge(
        token_df[["token_id", "token_winner", "final_price"]],
        on="token_id",
        how="inner",
    )
    if c.empty:
        return c

    c["dt"] = pd.to_datetime(c["block_timestamp"], unit="s", utc=True)
    c["final_value_usdc"] = c["quantity"] * c["final_price"]
    return c

# Group by tx_hash × wallet × side so that every on-chain transaction is one row
# per wallet-side pair, regardless of how many individual fills it contains.
GROUP_KEYS = ["tx_hash", "wallet", "side"]

READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "trade_date", "condition_id",
    "token_id", "outcome", "price", "quantity", "usdc_amount", "position",
    "wallet", "side",
]

grouped_parts: list[pd.DataFrame] = []
sample_fills = None

total_raw_rows = 0
total_filtered_rows = 0

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=READ_COLS)
    total_raw_rows += len(raw_chunk)

    enriched_chunk = enrich_trade_chunk(raw_chunk)
    total_filtered_rows += len(enriched_chunk)

    if sample_fills is None and len(enriched_chunk) > 0:
        sample_fills = enriched_chunk.head(5).copy()

    if len(enriched_chunk) == 0:
        continue

    # qty-weighted price numerator: sum(price * quantity) per group
    enriched_chunk["price_x_qty"] = enriched_chunk["price"] * enriched_chunk["quantity"]

    grouped_part = (
        enriched_chunk.groupby(GROUP_KEYS, sort=False)
        .agg(
            dt               = ("dt",             "first"),
            condition_id     = ("condition_id",   "first"),
            outcome          = ("outcome",         "first"),
            token_winner     = ("token_winner",    "first"),
            final_price      = ("final_price",     "first"),
            position         = ("position",        "max"),
            total_quantity   = ("quantity",        "sum"),
            price_x_qty_sum  = ("price_x_qty",    "sum"),
            trade_value_usdc = ("usdc_amount",     "sum"),
            final_value_usdc = ("final_value_usdc","sum"),
            num_fills        = ("log_index",       "count"),
        )
        .reset_index()
    )
    grouped_parts.append(grouped_part)

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed {i}/{len(trade_files)} shards | "
            f"raw rows: {total_raw_rows:,} | in-range rows: {total_filtered_rows:,}"
        )

if not grouped_parts:
    raise ValueError("No in-range trade rows after token filter")

grouped = pd.concat(grouped_parts, ignore_index=True)

# In case a key spans multiple shards, combine partial aggregates.
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum", "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "tx_hash", "side"], inplace=True, ignore_index=True)

print(
    f"Rows after market filter: {total_filtered_rows:,}  "
    f"(dropped {total_raw_rows - total_filtered_rows:,} rows outside [{START_DATE}, {END_DATE}])"
)
print(f"Prepared grouped rows: {len(grouped):,}")
if sample_fills is not None:
    sample_fills.head(5)


Processed 16/16 shards | raw rows: 93,357,019 | in-range rows: 1,995,789
Rows after market filter: 1,995,789  (dropped 91,361,230 rows outside [2026-01-01, 2026-01-05])
Prepared grouped rows: 1,648,559


## 3 · Enriched fill table

All fills are already filtered to in-range markets and carry resolution columns
from the inner join performed in the previous step.

| column | description |
|---|---|
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Settlement price: 1.0 if `token_winner=True`, 0.0 otherwise |
| `final_value_usdc` | `quantity × final_price` — USDC value of the position at settlement |

In [104]:
if sample_fills is None:
    print("No enriched fills available after filtering")
    pd.DataFrame(columns=[
        "wallet", "side", "condition_id", "dt", "outcome", "quantity", "price",
        "position", "usdc_amount", "token_winner", "final_price", "final_value_usdc"
    ])
else:
    print(f"Raw rows scanned across all shards: {total_raw_rows:,}")
    print(f"In-range rows after token join:     {total_filtered_rows:,}")
    print(f"token_winner null: {sample_fills['token_winner'].isna().sum():,}  (sample should be 0)")
    sample_fills[["wallet", "side", "condition_id", "dt", "outcome", "quantity", "price", "position",
                  "usdc_amount", "token_winner", "final_price", "final_value_usdc"]].head(5)


Raw rows scanned across all shards: 93,357,019
In-range rows after token join:     1,995,789
token_winner null: 0  (sample should be 0)


## 4 · Group by tx_hash × wallet × side

A single on-chain transaction can generate multiple fills across price levels.
Each row in `grouped` aggregates all fills for one wallet-side pair within one transaction:
- `num_fills` — number of individual fill events in the transaction
- `total_quantity` — sum of filled quantity
- `trade_value_usdc` — sum of USDC spent/received
- `avg_price` — quantity-weighted average fill price

In [105]:
print(f"Grouped rows: {len(grouped):,}")
grouped.head(10)


Grouped rows: 1,648,559


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,avg_price
0,0x69aabd6b1d5511f6a5c232eb4b3b84c8860aaefe98db11d11027caae3b8cfeb6,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,BUY,2026-01-03 09:41:21+00:00,0x6ed5173182f270baafbb61c3ca07eb709462f9eaa65431099f96698ec435321a,Blackhawks,True,1.0,9.729728,9.729728,3.599999,9.729728,1,0.370
1,0xc2f2ffa7741a34f50510ade4c4cdece31e12af38b45572061f55ad260f7ca0d9,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,BUY,2026-01-01 16:09:31+00:00,0x212542b6d0eb29dce20144d0aa9e3a024e83c6c4340fc6e21532cbaff2ac7cab,Blues,True,1.0,7.000000,7.000000,3.010000,7.000000,1,0.430
2,0x170ca343f7cafb6b9e03d5517c993cce0dd68e77542e20f5f1265895d3b05339,0x00002981563121d523c198f6fe26e1360d3f0fe3,BUY,2025-12-28 11:17:03+00:00,0x55b9037ea243c7b463fddf4607af375b71a57857fe366ea9194ab00c031dd703,Yes,True,1.0,5.000000,5.000000,4.950000,5.000000,1,0.990
3,0x4db6e15ef071774f3469691b153140cf2cdfbbf8c88a7e9080671f3682499cdd,0x00002981563121d523c198f6fe26e1360d3f0fe3,BUY,2026-01-02 07:29:03+00:00,0xc191f31b52374714923b0946d8c530b67194ad083a660dfd3c7049ac3b00b6b1,Yes,True,1.0,5.000000,5.000000,4.995000,5.000000,1,0.999
4,0xe72c689d717d6e5dc653835e3f40aaa4e340310ee3947610476ac1920f453b20,0x00002981563121d523c198f6fe26e1360d3f0fe3,BUY,2025-12-28 11:17:01+00:00,0x2bb7e3a417068b222bc3682278e27272088c4d570dd301af9c4f922a4e57071d,Yes,True,1.0,5.000000,5.000000,4.945000,5.000000,1,0.989
5,0xf5e6a6cd850f667066bf210a91d773226ba80af7e99ef96ebbbf68541387433f,0x00007bd051e6eb4e74933247c01559fa377740a5,BUY,2026-01-03 19:38:21+00:00,0x6f8e55dcb34c65b54196d7df2dcfbf2bd7eb4ade9c32b964a4e48328a34c37ed,Over,False,0.0,20.833332,20.833332,9.999999,0.000000,1,0.480
6,0x31a7b127084db38ab5d9fd124ad1cebb681620c5f5af09825808192203f83456,0x000142a0d4458481a5815a3e773e21989c59a06a,BUY,2026-01-03 16:30:41+00:00,0x68303d339f961812753a5af8899df01adfcad7380a2625fa7b944e2ae30f9683,Yes,False,0.0,7.058822,7.058822,5.999998,0.000000,1,0.850
7,0x9dc95c0550027f35e943e76d814760eb300d54eaf4f9b7b6b67c3bdbc5fbfafe,0x000142a0d4458481a5815a3e773e21989c59a06a,SELL,2026-01-03 17:00:27+00:00,0x68303d339f961812753a5af8899df01adfcad7380a2625fa7b944e2ae30f9683,Yes,False,0.0,0.008822,7.050000,0.987000,0.000000,1,0.140
8,0x13358c2c4e8b431eb2ffae69ad51fee1ee2493d60a9b5854ce24dc6ba30f3464,0x00090e8b4fa8f88dc9c1740e460dd0f670021d43,BUY,2026-01-03 23:06:43+00:00,0xe377cc3f81cabf05e05be23be9be14a889c34f35eb38e6d166da4448d4b7850c,Yes,True,1.0,1964.591533,1153.846152,1138.846153,1153.846152,1,0.987
9,0x257f1f480aefa59e542624e623851c8bd146c58557308ac09b592a3663e4a57d,0x00090e8b4fa8f88dc9c1740e460dd0f670021d43,BUY,2026-01-03 23:01:37+00:00,0xe377cc3f81cabf05e05be23be9be14a889c34f35eb38e6d166da4448d4b7850c,Yes,True,1.0,25.852306,25.852306,25.516227,25.852306,1,0.987


## 5 · Per-trade P&L and wallet summary (training data only)

P&L is calculated **per trade** based on whether the traded token is a winner:

| side | `trade_pnl` formula | intuition |
|---|---|---|
| BUY | `final_value_usdc − trade_value_usdc` | bought tokens; profit if `final_price > entry_price` |
| SELL | `trade_value_usdc − final_value_usdc` | sold tokens; profit if sold above final settlement price |

where `final_value_usdc = total_quantity × final_price` (1.0 if the token is a winner, 0.0 otherwise).

Wallet P&L = `Σ trade_pnl` over training trades.

In [106]:
# ── per-trade P&L: final_value vs. trade_value, signed by side ──────────────
# BUY:  wallet paid trade_value_usdc, ends up with tokens worth final_value_usdc
#       → pnl = final_value_usdc - trade_value_usdc  (positive if token wins)
# SELL: wallet received trade_value_usdc, gave up tokens worth final_value_usdc
#       → pnl = trade_value_usdc - final_value_usdc  (positive if token loses)
is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = (
    (grouped["final_value_usdc"] - grouped["trade_value_usdc"]).where(is_buy,
    grouped["trade_value_usdc"] - grouped["final_value_usdc"])
)

# ── restrict to training period for wallet ranking ────────────────────────────
end_train_ts   = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train  = grouped[grouped["dt"] < end_train_ts]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets      = ("condition_id",   "nunique"),
        num_trades       = ("num_fills",       "sum"),
        total_cost_usdc  = ("trade_value_usdc", "sum"),
        pnl_usdc         = ("trade_pnl",       "sum")
                )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.head(5)

Unique wallets (training period): 36,436


,wallet,num_markets,num_trades,total_cost_usdc,pnl_usdc
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,36,2208,2.482478e+06,1.306898e+06
1,0x0d3b10b8eac8b089c6e4a695e65d8e044167c46b,30,8912,6.604998e+06,8.269899e+05
2,0x492442eab586f242b53bda933fd5de859c8a3782,34,1556,3.065441e+06,5.452317e+05
3,0xe74a4446efd66a4de690962938f550d8921a40ee,8,403,6.914630e+05,3.993424e+05
4,0x37e4728b3c4607fb2b3b205386bb1d1fb1a8c991,41,668,1.452030e+06,3.985996e+05


## 6 · Market-level volume summary

In [107]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum")
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 5,809


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x386c70eb5b79e8d6b05bc371033387f0b9a5d93180429dfc6f13fc25dd526adb,Seahawks vs. 49ers,2026-01-04 00:00:00+00:00,2629,23564,7.344102e+06,7.601299e+06
1,0x6fe48420bd2c25a2594d158487b08fbc12edd080ff8c3823566bc49fdc2e5b5c,Ravens vs. Steelers,2026-01-04 00:00:00+00:00,2580,26280,6.771192e+06,6.671726e+06
2,0x68730687766f5294a9840b6d3b7b57b8040d082dd69cffbcff899610decb8e17,Oregon vs. Texas Tech Red Raiders,2026-01-01 00:00:00+00:00,1428,13079,5.240909e+06,5.158345e+06
3,0x39310dbdbc542eda4650a85fb8121fad8235485a4c46766fbf8d8551f2b23936,Thunder vs. Warriors,2026-01-03 00:00:00+00:00,1907,15671,4.606402e+06,4.792603e+06
4,0xa6535047a0fef371d1d1b44693508960eef82c53db3e311df0ad5c6b650d810f,Panthers vs. Buccaneers,2026-01-04 00:00:00+00:00,1917,13511,4.314822e+06,4.267597e+06
5,0xc48fcf49b52f5687875d5636f718ff726a18a108c086cfc726eabf130eacc3e3,Knicks vs. Spurs,2026-01-01 00:00:00+00:00,1973,20293,4.179168e+06,4.137244e+06
6,0x269b70d13789a587fde5b6482688ae5bb18c8de7e357ec5903c32948454ea849,Ole Miss vs. Georgia,2026-01-02 00:00:00+00:00,1505,19437,4.110967e+06,4.121814e+06
7,0x88a945f5acfae0e9f2ad6af05819554212da4f9e36c1310641967bd09e66a265,Timberwolves vs. Heat,2026-01-03 00:00:00+00:00,2762,16921,3.845592e+06,3.973347e+06
8,0xe1c1cfe25fa7d982a943264579081767708594952ea53ba3db8888d6fa79d853,Pelicans vs. Bulls,2026-01-01 00:00:00+00:00,2176,14986,3.756132e+06,3.756625e+06
9,0x9b2aad260f6b5dea10af0b03d5e4788adf261f0eea9db9616a3581e06bdaef0e,Grizzlies vs. Lakers,2026-01-05 00:00:00+00:00,2724,23818,3.660457e+06,3.806652e+06


## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets (training period).

In [108]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc,wallet_count_complement
quantile,,,,,
0.001,36,1.0,1.0,-47257.59,36400
0.01,364,1.0,1.0,-3721.89,36072
0.05,1822,1.0,1.0,-496.63,34614
0.1,3644,1.0,1.0,-140.59,32792
0.25,9109,1.0,1.0,-13.64,27327
0.5,18218,3.0,2.0,0.00,18218
0.75,27327,12.0,5.0,7.00,9109
0.9,32792,36.0,12.0,96.27,3644
0.95,34614,81.0,21.0,373.49,1822


## 8 · Full enriched trade table

One row per tx_hash × wallet × side, with all enrichment columns.

In [109]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,num_fills
0,0x69aabd6b1d5511f6a5c232eb4b3b84c8860aaefe98db11d11027caae3b8cfeb6,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,BUY,2026-01-03 09:41:21+00:00,0x6ed5173182f270baafbb61c3ca07eb709462f9eaa65431099f96698ec435321a,Blackhawks,9.729728,9.729728,0.370,True,1.0,3.599999,9.729728,6.129729,1
1,0xc2f2ffa7741a34f50510ade4c4cdece31e12af38b45572061f55ad260f7ca0d9,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,BUY,2026-01-01 16:09:31+00:00,0x212542b6d0eb29dce20144d0aa9e3a024e83c6c4340fc6e21532cbaff2ac7cab,Blues,7.000000,7.000000,0.430,True,1.0,3.010000,7.000000,3.990000,1
2,0x170ca343f7cafb6b9e03d5517c993cce0dd68e77542e20f5f1265895d3b05339,0x00002981563121d523c198f6fe26e1360d3f0fe3,BUY,2025-12-28 11:17:03+00:00,0x55b9037ea243c7b463fddf4607af375b71a57857fe366ea9194ab00c031dd703,Yes,5.000000,5.000000,0.990,True,1.0,4.950000,5.000000,0.050000,1
3,0x4db6e15ef071774f3469691b153140cf2cdfbbf8c88a7e9080671f3682499cdd,0x00002981563121d523c198f6fe26e1360d3f0fe3,BUY,2026-01-02 07:29:03+00:00,0xc191f31b52374714923b0946d8c530b67194ad083a660dfd3c7049ac3b00b6b1,Yes,5.000000,5.000000,0.999,True,1.0,4.995000,5.000000,0.005000,1
4,0xe72c689d717d6e5dc653835e3f40aaa4e340310ee3947610476ac1920f453b20,0x00002981563121d523c198f6fe26e1360d3f0fe3,BUY,2025-12-28 11:17:01+00:00,0x2bb7e3a417068b222bc3682278e27272088c4d570dd301af9c4f922a4e57071d,Yes,5.000000,5.000000,0.989,True,1.0,4.945000,5.000000,0.055000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1648554,0xf94ea779ee206bc3fad7aa6aaeaf18f2bf11315420f25f6381bbb6f2eef956d4,0xfffb0cd89d8b124b3081011db7e7c47676ff5f95,BUY,2026-01-03 04:31:11+00:00,0xfa6e92dadadedf14750763507f107b6b633b653c29c4c64f392a1428b8cfe131,Lakers,60.000000,60.000000,0.840,True,1.0,50.400000,60.000000,9.600000,1
1648555,0xfa5422f2f401435107f3bc5e85c02e2e97d10481822efb8c47c5cf87699c2e3d,0xfffb0cd89d8b124b3081011db7e7c47676ff5f95,BUY,2026-01-01 00:57:27+00:00,0xc48fcf49b52f5687875d5636f718ff726a18a108c086cfc726eabf130eacc3e3,Spurs,73.000000,18.830000,0.450,True,1.0,8.473500,18.830000,10.356500,1
1648556,0x7dc4ece497cce7227abe0e5ede894c1094a2610930be33f3e46cee2d4117850c,0xfffc580e5fe57b33140a23f56fd7ca22141d088c,BUY,2026-01-05 09:15:51+00:00,0xd738e71656c315e90cc6712b6afc534724bcd6bf6f40b03f194029511f117369,No,17.011763,17.011763,0.850,True,1.0,14.459998,17.011763,2.551765,1
1648557,0xe82f2958ef28d355a5e3542dc813615c9365886d0157ae84c63c9eed5d48f9d5,0xfffdeb2df30997713b0186c8cb26c6b75c69b7c4,BUY,2026-01-03 12:08:13+00:00,0x386c70eb5b79e8d6b05bc371033387f0b9a5d93180429dfc6f13fc25dd526adb,Seahawks,350.000000,350.000000,0.570,True,1.0,199.500000,350.000000,150.500000,1


## 9 · Export top-5% wallets to parquet shards

Identifies top-5% wallets by P&L on the **training period**, then filters
the already-grouped `(tx_hash, wallet, side)` DataFrame to those wallets and
writes the result to `data/polygon_trades_processed/*.parquet`.

Each output row is one **transaction** (not a fill), with aggregated columns:
- `total_quantity` — sum of filled shares
- `avg_price` — quantity-weighted average fill price
- `trade_value_usdc` — total USDC spent/received
- `num_fills` — number of individual fill events in the transaction
- `is_train` — `True` if `dt.date <= END_DATE_TRAIN`


In [110]:
# ── identify top-5% wallets by training-period P&L ───────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:                {len(top_wallets):,}")

# print deciles of training P&L for top wallets
top_pnl = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"]
for i in range(0, 11):
    q = i / 10
    print(f"  P&L at {q:.0%} pct of top wallets: {top_pnl.quantile(q):,.2f} USDC")

P&L threshold (95th pct, training): 373.49 USDC
Top-5% wallet count:                1,822
  P&L at 0% pct of top wallets: 373.95 USDC
  P&L at 10% pct of top wallets: 451.59 USDC
  P&L at 20% pct of top wallets: 548.16 USDC
  P&L at 30% pct of top wallets: 702.01 USDC
  P&L at 40% pct of top wallets: 881.42 USDC
  P&L at 50% pct of top wallets: 1,082.41 USDC
  P&L at 60% pct of top wallets: 1,467.15 USDC
  P&L at 70% pct of top wallets: 1,938.75 USDC
  P&L at 80% pct of top wallets: 2,966.46 USDC
  P&L at 90% pct of top wallets: 6,462.03 USDC
  P&L at 100% pct of top wallets: 1,306,898.30 USDC


In [111]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── filter grouped to top-wallet rows and stamp is_train ─────────────────────
top_grouped = grouped[grouped["wallet"].isin(top_wallets)].copy()
top_grouped["is_train"] = top_grouped["dt"].dt.date <= END_DATE_TRAIN

export_df = top_grouped[EXPORT_COLS].reset_index(drop=True)

# write as a single shard (mirrors the single-shard input)
out_file = OUT_DIR / "0.parquet"
export_df.to_parquet(out_file, index=False)
output_files = [out_file]

top_trades_preview = export_df.head(5).copy()
top_trades_count   = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows for top wallets: {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
print(f"\nSaved partitioned output → {OUT_DIR}")


Total grouped rows for top wallets: 516,883
  training rows: 327,240
  test rows:     189,643
Output shards:  1

Saved partitioned output → ../../data/polygon_trades_processed


In [112]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.
It also reports whether the exchange appears in the currently filtered dataset
(appearance depends on the chosen date range).


In [113]:
# ── Unit test: CTF Exchange contract is not selected as a top wallet ──────────
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

# Presence in grouped depends on date filtering; this is informational.
exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

# Must always hold regardless of date range.
assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")


INFO  CTF Exchange rows not present in current filtered dataset
PASS  CTF Exchange not in top_wallets: top_wallets has 1,822 entries
